In [3]:
import pandas as pd

df = pd.read_csv('C:/ecommerce-churn-analysis/data/raw/data.csv', encoding='ISO-8859-1')

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [4]:
print("--- ИНФОРМАЦИЯ О ДАТАСЕТЕ ---")
print(df.info())

print("\n--- ПРОПУСКИ В КОЛОНКАХ ---")
print(df.isnull().sum())

print("\n---- ДУБЛИКАТЫ ---")
print(f"Полных дубликатов строк: {df.duplicated().sum()}")

--- ИНФОРМАЦИЯ О ДАТАСЕТЕ ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
None

--- ПРОПУСКИ В КОЛОНКАХ ---
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

---- ДУБЛИКАТЫ ---
Полных дубликатов строк: 5268


In [5]:
print("Минимальное количество:", df['Quantity'].min())
print("Минимальная цена:", df['UnitPrice'].min())

df[df['Quantity'] < 0].head()

Минимальное количество: -80995
Минимальная цена: -11062.06


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,12/1/2010 9:41,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,12/1/2010 9:49,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,12/1/2010 10:24,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,12/1/2010 10:24,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,12/1/2010 10:24,0.29,17548.0,United Kingdom


## Decision Log: Очистка и предобработка данных

На основе первичного анализа (Data Quality Check) приняты следующие решения по обработке аномалий:

1. **Удаление дубликатов** 
    - *Проблема:* Обнаружено 5 268 полных дубликатов строк.
    - *Решение:* Удаляем их, так как это технический сбой дублирования тразакций при записи в базу.

2. **Обработка пропусков в CustomerID:** 
    - *Проблема:* ~135тыс. тразакций не имеют ID клиента ('CustomerID is null').
    - *Решение:* Создаем два среза данных:
        - 'df_clean': основной датасет БЕЗ анонимных покупок (нужен для RFM-анализа и расчета оттока Churn, где важен конкретный клиент).
        - 'df_raw_sales': датасет с сохранением всех продаж (понадобится, если захотим посчитать общую выручку магазина)
3. **Фильтрация возвратов (Quantity < 0):**
    - *Проблема:* Транзакция с отрицательным количеством представляют собой отмены чеков ('InvoiceNo' начинается на 'C').
    - *Решение:* Исключаем возвраты из основного датасета продаж ('Quantity > 0').
4. **Фильтрация аномальных цен (UnitPrice <= 0):**
    - *Проблема:* Отрицательные и нулевые цена - это корректировки бухгалтерии и списания.
    - *Решение:* Оставляем только транзакции с корректной ценой ('UnitPrice >0').
5. **Приведение типов данных:** 
    - *Проблема:* Столбец 'InvoiceDate' имеет текстовый тип ('object').
    - *Решение:* Переводим в формат 'datetime64' для работы со временными рядами.

In [10]:
df_clean = df.drop_duplicates().copy()

df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

df_clean = df_clean.dropna(subset=['CustomerID'])

df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print(f"Размер сырого датасета: {df.shape[0]} строк")
print(f"Размер очищенного датасета: {df_clean.shape[0]} строк")
print(f"Удалено/отфильтровано: {df.shape[0] - df_clean.shape[0]} строк (аномалии, возвраты, пропуски)")

df_clean.head()

Размер сырого датасета: 541909 строк
Размер очищенного датасета: 392692 строк
Удалено/отфильтровано: 149217 строк (аномалии, возвраты, пропуски)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [14]:
df_clean['TotalSum'] = df_clean['Quantity'] * df_clean['UnitPrice']

df_clean.to_csv('C:/ecommerce-churn-analysis/data/processed/clean_data.csv', index=False)

print("--- БАЗОВЫЕ МЕТРИКИ МАГАЗИНА ---")
print(f"Общая выручка (Total Revenue): {df_clean['TotalSum'].sum():,.2f}")
print(f"Уникальных клиентов: {df_clean['CustomerID'].nunique():,}")
print(f"Всего совершенных заказов: {df_clean['InvoiceNo'].nunique():,}")

--- БАЗОВЫЕ МЕТРИКИ МАГАЗИНА ---
Общая выручка (Total Revenue): 8,887,208.89
Уникальных клиентов: 4,338
Всего совершенных заказов: 18,532
